# Diagnostic - AUC-PR proche du hasard

À lancer AVANT de relancer comparaison_modeles.py, pour identifier la
cause d'un AUC-PR anormalement bas (~0.02, proche du taux de fraude
de base, ce qui signale une absence quasi totale de signal détecté).

Auteur : Rasmané

In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import average_precision_score
from sklearn.tree import DecisionTreeClassifier

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 150)

DOSSIER = "C:/Users/hp/Desktop/Formation FORCE certificats Intelligence Artificielle/Projet_final/data"

X_train = pd.read_csv(f"{DOSSIER}/X_train.csv")
X_val = pd.read_csv(f"{DOSSIER}/X_val.csv")
y_train = pd.read_csv(f"{DOSSIER}/y_train.csv").squeeze()
y_val = pd.read_csv(f"{DOSSIER}/y_val.csv").squeeze()

## 1. VÉRIFICATIONS DE BASE (alignement, forme, dtypes)

In [ ]:
print("=" * 70)
print("1. VÉRIFICATIONS DE BASE")
print("=" * 70)
print(f"X_train : {X_train.shape} | y_train : {y_train.shape}")
print(f"X_val   : {X_val.shape} | y_val   : {y_val.shape}")
print(f"Taux de fraude train : {y_train.mean()*100:.4f}%")
print(f"Taux de fraude val   : {y_val.mean()*100:.4f}%")
print(f"\nColonnes de X_train ({len(X_train.columns)}) :")
print(X_train.columns.tolist())
print(f"\nDtypes :")
print(X_train.dtypes)

# Valeurs manquantes / infinies introduites par les ratios
print(f"\nValeurs manquantes dans X_train : {X_train.isnull().sum().sum()}")
nb_inf = np.isinf(X_train.select_dtypes(include=[np.number])).sum().sum()
print(f"Valeurs infinies dans X_train : {nb_inf}")
if nb_inf > 0:
    colonnes_avec_inf = X_train.columns[
        np.isinf(X_train.select_dtypes(include=[np.number])).any()
    ]
    print(f"Colonnes concernées : {colonnes_avec_inf.tolist()}")
    print("-> À nettoyer (clip ou remplacement) avant tout entraînement.")

## 2. CORRÉLATION DE CHAQUE FEATURE AVEC LA CIBLE (SUR LE TRAIN)

In [ ]:
print("\n" + "=" * 70)
print("2. CORRÉLATION DE CHAQUE FEATURE AVEC isFraud (train)")
print("=" * 70)
correlations = X_train.corrwith(y_train).sort_values(key=abs, ascending=False)
print(correlations)
print("\n-> Si TOUTES les corrélations sont proches de 0 (< 0.02 en valeur")
print("   absolue), c'est cohérent avec un AUC-PR proche du hasard : les")
print("   features n'ont individuellement quasi aucun pouvoir discriminant")
print("   linéaire. Passe au point 3 pour tester le pouvoir non-linéaire.")

## 3. POUVOIR DISCRIMINANT NON-LINÉAIRE (arbre de décision simple)

In [ ]:
# Un arbre peu profond capture des seuils/interactions simples.
# S'il n'arrive toujours pas à dépasser le hasard, le signal est
# probablement quasi absent dans les données telles quelles.
print("\n" + "=" * 70)
print("3. TEST RAPIDE AVEC UN ARBRE DE DÉCISION PEU PROFOND")
print("=" * 70)

arbre = DecisionTreeClassifier(max_depth=5, class_weight="balanced", random_state=42)
arbre.fit(X_train, y_train)
proba_arbre = arbre.predict_proba(X_val)[:, 1]
auc_pr_arbre = average_precision_score(y_val, proba_arbre)
print(f"AUC-PR avec un arbre simple (profondeur 5) : {auc_pr_arbre:.4f}")

importance_features = pd.Series(
    arbre.feature_importances_, index=X_train.columns
).sort_values(ascending=False)
print("\nImportance des features pour cet arbre :")
print(importance_features[importance_features > 0])

## 4. RÉFÉRENCE : PERFORMANCE D'UN SEUL SIGNAL CONNU (isFlaggedFraud)

In [ ]:
print("\n" + "=" * 70)
print("4. RÉFÉRENCE - isFlaggedFraud utilisé seul comme score")
print("=" * 70)
if "isFlaggedFraud" in X_val.columns:
    auc_pr_flag_seul = average_precision_score(y_val, X_val["isFlaggedFraud"])
    print(f"AUC-PR avec isFlaggedFraud seul : {auc_pr_flag_seul:.4f}")
else:
    print("isFlaggedFraud absent de X_val (probablement retiré comme feature -")
    print("à vérifier si c'était volontaire).")

## 5. RÉFÉRENCE : PERFORMANCE DE amount SEUL

In [ ]:
print("\n" + "=" * 70)
print("5. RÉFÉRENCE - amount utilisé seul comme score")
print("=" * 70)
if "amount" in X_val.columns:
    auc_pr_amount_seul = average_precision_score(y_val, X_val["amount"])
    print(f"AUC-PR avec amount seul : {auc_pr_amount_seul:.4f}")

## 6. VÉRIFICATION DE LA COHÉRENCE isFraud DANS L'ÉCHANTILLON D'ORIGINE

In [ ]:
print("\n" + "=" * 70)
print("6. RAPPEL - à recroiser avec ton EDA (axe 3 / axe 4)")
print("=" * 70)
print("Reprends les résultats de eda_echantillon.py :")
print("- Axe 3, point 3.4 (pattern solde destinataire figé) : le ratio")
print("  taux_masque / taux_hors_masque était-il nettement > 1 ?")
print("- Axe 4, point 4.2 (corrélation avec isFraud) : quelles étaient les")
print("  valeurs exactes ? Si elles étaient déjà proches de 0 dans l'EDA,")
print("  le problème n'est pas un bug de ce script mais une caractéristique")
print("  réelle de ce dataset synthétique (le générateur Cifer/SDV ne")
print("  préserve peut-être pas les patterns déterministes du PaySim")
print("  original).")

print("\n" + "=" * 70)
print("DIAGNOSTIC TERMINÉ")
print("=" * 70)
print("Interprétation :")
print("- Si l'arbre simple (point 3) obtient un AUC-PR nettement meilleur")
print("  (> 0.3-0.5) que XGBoost/LightGBM (0.024), le problème vient du")
print("  PIPELINE d'entraînement (mauvais réglage, pas du manque de signal).")
print("- Si l'arbre simple obtient AUSSI un score proche du hasard, le signal")
print("  est probablement quasi absent des features actuelles -> il faudra")
print("  soit créer des features plus fines, soit documenter cette limite")
print("  du dataset synthétique dans le rapport (point 19 du sujet : biais")
print("  et limites des données synthétiques).")